# 🌐 Lab 2: Atmospheric Physics, ISA 1976 & Dryden Turbulence
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/blaze505050/drone-digital-twin/blob/main/examples/labs/Lab2_Atmospheric_Physics_and_Turbulence.ipynb)

Welcome to **Lab 2 of the DronePy Aerospace & Robotics Curriculum**!
In this lab, you will study the thermodynamic impact of altitude on atmospheric pressure, temperature, and air density, and simulate vehicle flight dynamics under **MIL-F-8785C Dryden continuous turbulence** and **discrete 1-cosine gusts**.

---
### 🎯 Learning Objectives
1. Model the International Standard Atmosphere (ISA 1976) up to 11,000 meters.
2. Analyze how rotor thrust decays with altitude due to reduced air density $\rho(h)$.
3. Implement continuous stochastic Dryden turbulence and discrete wind gusts.
4. Measure vehicle attitude error and control effort in turbulent mountain pass conditions.


In [ ]:
# Setup dependencies
try:
    import dronepy
except ImportError:
    !pip install -q git+https://github.com/blaze505050/drone-digital-twin.git
    import dronepy

import numpy as np
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

print(f"DronePy Version: {dronepy.__version__}")


---
## 1. Mathematical Theory: ISA 1976 Atmosphere

The standard atmosphere model in the troposphere ($0 \le h \le 11{,}000\text{ m}$) is governed by:
- Temperature lapse: $T(h) = T_0 + L \cdot h$, where $T_0 = 288.15\text{ K}$, $L = -0.0065\text{ K/m}$
- Barometric pressure equation:
  $$P(h) = P_0 \left(1 + \frac{L \cdot h}{T_0}\right)^{-\frac{g_0 M}{R \cdot L}}$$
- Air density (ideal gas law):
  $$\rho(h) = \frac{P(h)}{R_{\text{specific}} \cdot T(h)}$$

Rotor thrust scales linearly with local density:
$$T(h) = C_T \cdot \rho(h) \cdot n^2 D^4$$
At higher altitudes, a multirotor must spin its motors significantly faster to generate the same hover thrust, consuming more electrical power and reducing control margins.


In [ ]:
# 2. Investigating Altitude Density Lapse
altitudes_m = np.linspace(0, 4000, 9)
print(f"{'Altitude (m)':>12} | {'Temp (C)':>10} | {'Pressure (hPa)':>14} | {'Density (kg/m3)':>16} | {'Thrust %':>10}")
print("-" * 70)

env_sl = dronepy.Environment.standard_atmosphere(altitude=0.0)
rho_0 = env_sl.density_kgm3

for h in altitudes_m:
    env = dronepy.Environment.standard_atmosphere(altitude=h)
    rho = env.density_kgm3
    p = env.pressure_pa / 100.0  # hPa
    t_c = env.temperature_k - 273.15
    thrust_pct = (rho / rho_0) * 100.0
    print(f"{h:12.0f} | {t_c:10.2f} | {p:14.2f} | {rho:16.4f} | {thrust_pct:9.1f}%")


---
## 3. Wind & Turbulence Modeling: 1-Cosine Discrete Gusts
DronePy supports both continuous Dryden turbulence and discrete 1-cosine gusts:
$$v_{\text{gust}}(t) = \begin{cases} 
0 & t < t_{\text{start}} \\
\frac{V_{\text{max}}}{2} \left(1 - \cos\frac{2\pi (t - t_{\text{start}})}{T_{\text{duration}}}\right) & t_{\text{start}} \le t \le t_{\text{start}} + T_{\text{duration}} \\
0 & t > t_{\text{start}} + T_{\text{duration}}
\end{cases}$$

Let us simulate a drone flight subjected to a crosswind gust at $t = 1.0\text{ s}$.


In [ ]:
# Configure vehicle and environment with a 6 m/s 1-cosine crosswind gust
drone = dronepy.Drone.quadcopter(mass=1.5)
gust = dronepy.Wind.gust(
    magnitude=6.0,
    direction_deg=90.0, # East wind
    start_time=1.0,
    duration=1.5
)
env_gust = dronepy.Environment.standard_atmosphere(altitude=0.0, wind=gust)

# Run 4-second 6-DOF simulation
flight = dronepy.Flight(drone=drone, environment=env_gust, duration=4.0)
res = flight.result

print(f"Simulation completed with {len(res.time)} timesteps.")
print(f"Max East Drift:     {np.max(np.abs(res.pos_ned[:, 1])):.3f} m")
print(f"Max Roll Excursion: {np.max(np.abs(res.euler_deg[:, 0])):.2f} deg")


---
## 📝 Student Exercise: High-Altitude Mountain Pass Stability

### Scenario:
A reconnaissance drone is deployed in the Himalayas at an elevation of **$h = 3{,}200\text{ m}$ MSL**.
At $t = 1.5\text{ s}$, the drone encounters a crosswind gust of **$V_{\text{max}} = 8.5\text{ m/s}$** lasting for $2.0\text{ s}$.

### Your Tasks:
1. Initialize an `Environment` at $3{,}200\text{ m}$ elevation using `Environment.standard_atmosphere(altitude=3200.0, wind=...)`.
2. Run a 4-second `Flight` simulation of a $1.5\text{ kg}$ quadcopter.
3. Determine:
   - The air density ratio $\rho(3200) / \rho(0)$
   - The peak motor RPM reached during gust rejection
   - The maximum horizontal displacement from the hover setpoint


In [ ]:
# ══════════════════════════════════════════════════════════════════
# STUDENT SOLUTION CELL - Execute the mountain pass simulation:
# ══════════════════════════════════════════════════════════════════
alt_himalayas = 3200.0
himalaya_gust = dronepy.Wind.gust(
    magnitude=8.5,
    direction_deg=90.0,
    start_time=1.5,
    duration=2.0
)
env_himalaya = dronepy.Environment.standard_atmosphere(altitude=alt_himalayas, wind=himalaya_gust)
drone_hi = dronepy.Drone.quadcopter(mass=1.5)

flight_hi = dronepy.Flight(drone=drone_hi, environment=env_himalaya, duration=4.0)
res_hi = flight_hi.result

rho_himalaya = env_himalaya.density_kgm3
rho_ratio = rho_himalaya / 1.225
peak_rpm = np.max(res_hi.motor_rpms)
max_drift = np.max(np.linalg.norm(res_hi.pos_ned[:, :2], axis=1))

print(f"Density Ratio:     {rho_ratio:.3f} ({rho_himalaya:.3f} kg/m3)")
print(f"Peak Motor RPM:    {peak_rpm:.1f} RPM")
print(f"Max Drift Radius:  {max_drift:.3f} m")

# Verification Assertions
assert rho_ratio < 0.78, "Density ratio should be ~0.72 at 3200m"
assert peak_rpm > 5000.0, "Motors must spin up to counter both thin air and gust"
assert max_drift > 0.1, "Drone should experience measurable drift before PID correction"
print("SUCCESS: Lab 2 atmospheric simulation verified successfully!")
